# 03 — Model Training & Comparison

Train and compare multiple machine learning models on the prepared Telco Customer Churn dataset.

**Workflow:** Load Data → Train Models → Evaluate → Compare → Select Best Model → Save

In [1]:
# ==========================================
# IMPORTS
# ==========================================

import pandas as pd
import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

## 1. Load Prepared Train-Test Data

In [2]:
# Load prepared train-test datasets

X_train = pd.read_csv('../data/processed/X_train.csv')
X_test = pd.read_csv('../data/processed/X_test.csv')

y_train = pd.read_csv('../data/processed/y_train.csv').squeeze()
y_test = pd.read_csv('../data/processed/y_test.csv').squeeze()

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (5634, 38)
X_test : (1409, 38)
y_train: (5634,)
y_test : (1409,)


## 2. Define Candidate Models

In [3]:
# Define candidate models

models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000,
        random_state=42
    ),

    'Random Forest': RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ),

    'XGBoost': XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric='logloss'
    )
}

## 3. Train Candidate Models

Train each candidate model using the same training dataset.

The test dataset is kept separate and is used only for model performance measurement.

In [4]:
# Train each candidate model

trained_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    trained_models[name] = model
    
    print(f"{name} trained successfully.")

/opt/anaconda3/envs/data/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Logistic Regression trained successfully.
Random Forest trained successfully.
XGBoost trained successfully.


## 5. Generate Test Predictions

Generate both:

- **Class predictions** for classification metrics
- **Probability predictions** for ROC-AUC

Probability predictions are particularly important because ROC-AUC evaluates how well the model ranks churners above non-churners.

In [5]:
# Generate predictions for each model

predictions = {}
probabilities = {}

for name, model in trained_models.items():
    predictions[name] = model.predict(X_test)
    probabilities[name] = model.predict_proba(X_test)[:, 1]

## 6. Model Performance Comparison

Compare the candidate models using multiple classification metrics.

ROC-AUC is used as the primary ranking metric, while Accuracy, Precision,
Recall, and F1-score provide additional information about classification performance.

In [6]:
# Calculate evaluation metrics for each model

results = []

for name in trained_models:

    y_pred = predictions[name]
    y_prob = probabilities[name]

    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_prob)
    })

results_df = pd.DataFrame(results)

# Rank models by ROC-AUC
results_df = results_df.sort_values(
    by='ROC-AUC',
    ascending=False
).reset_index(drop=True)

results_df

,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,Logistic Regression,0.804826,0.655172,0.558824,0.603175,0.842807
1,XGBoost,0.801278,0.653595,0.534759,0.588235,0.839841
2,Random Forest,0.784244,0.616667,0.494652,0.548961,0.818522


## 7. Select the Best Candidate

Select the model with the highest ROC-AUC score as the initial best candidate.

ROC-AUC is used here because it evaluates the model's ability to distinguish
between customers who churn and customers who remain.

In [7]:
# Select the highest-performing model based on ROC-AUC

best_model_name = results_df.loc[
    results_df['ROC-AUC'].idxmax(),
    'Model'
]

best_model = trained_models[best_model_name]

print(f"Selected model: {best_model_name}")
print(f"ROC-AUC: {results_df.loc[results_df['Model'] == best_model_name, 'ROC-AUC'].iloc[0]:.4f}")

Selected model: Logistic Regression
ROC-AUC: 0.8428


## 8. Best Model Summary

Display the complete metric profile of the selected model.

In [8]:
# Display best model metrics

best_results = results_df[
    results_df['Model'] == best_model_name
]

best_results

,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,Logistic Regression,0.804826,0.655172,0.558824,0.603175,0.842807


## 9. Save Modeling Results

Save the model comparison table so the experiment results remain reproducible
and can be referenced in the project report.

In [9]:
import os

os.makedirs('../reports', exist_ok=True)

results_df.to_csv(
    '../reports/model_comparison.csv',
    index=False
)

print("Model comparison saved to ../reports/model_comparison.csv")

Model comparison saved to ../reports/model_comparison.csv


## 10. Modeling Summary

Three classification models were trained and compared on the prepared
Telco Customer Churn dataset.

The model with the highest ROC-AUC on the held-out test set was selected
as the best-performing candidate and saved for detailed evaluation.

**Next:** `04_evaluation.ipynb`

In [10]:
print("=" * 50)
print("MODELING COMPLETE")
print("=" * 50)

print(f"Selected model : {best_model_name}")
print(f"ROC-AUC        : {results_df.loc[0, 'ROC-AUC']:.4f}")
print("=" * 50)

MODELING COMPLETE
Selected model : Logistic Regression
ROC-AUC        : 0.8428
